# Lab 4 — RAG with LangChain (EP10)

In the last three labs we wrote every piece of RAG ourselves. This lab builds **the same one** with LangChain and pairs each piece with our code

**What you'll do**
1. `Document` and `RecursiveCharacterTextSplitter` — and what the book's Listing 16.2 gets wrong
2. `OpenAIEmbeddings` + `InMemoryVectorStore` = Lab 1's `embed()` + `search()`
3. Our BM25 inside LangChain + `EnsembleRetriever` = Lab 2's `rrf()`
4. `ParentDocumentRetriever` = Lab 3's parent-child
5. Join everything into a chain and ask Gemma

**A note on versions:** the book uses `langchain.text_splitter` and `langchain.retrievers`, which moved in LangChain 1.x.
This lab uses the new locations, and doesn't use `langchain-community`, which is being sunset

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §16.1–16.4 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **5 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep10-lab4-langchain-rag.en.ipynb`** (the same code as in the video).

## 0. Setup

The same as Lab 1: Python 3.10+ or Colab, and `OPENROUTER_API_KEY` (Colab: 🔑 Secrets · your own machine: a `.env` file).
One key covers both Gemma and the embedding model — the whole lab costs under 1 US cent.

We pin LangChain's versions, because LangChain changes fast — even the book's import paths have moved

In [ ]:
%pip install -q "openai>=2.45" python-dotenv "pypdf>=6" numpy "tokenizers==0.23.3"
%pip install -q "langchain-core==1.7.6" "langchain-openai==1.7.4" "langchain-text-splitters==1.1.3" "langchain-classic==1.0.8"

In [ ]:
import os, re, math, urllib.request
from collections import Counter
import numpy as np
from pypdf import PdfReader
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False}, "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}
OPENROUTER = "https://openrouter.ai/api/v1"
tok = Tokenizer.from_pretrained(MODEL)

def n_tokens(text):
    return len(tok.encode(text, add_special_tokens=False).ids)

urllib.request.urlretrieve("https://arxiv.org/pdf/2606.24937v1", "hitchhikers-guide-v1.pdf")

TESTS = [
    ("What is PLAID?", {301}),
    ("What is HyDE?", {303}),
    ("What does FAISS stand for?", {298}),
    ("How can I search millions of vectors quickly?", {298}),
    ("Which method compares every query token with every document token?", {300}),
    ("How do I make keyword search understand synonyms?", {299}),
    ("When should I fine-tune a model instead of looking things up?", {296}),
    ("Which chunks should I give the model if small ones retrieve better but lack context?", {302, 303}),
    ("How do I combine keyword search and embedding search?", {298}),
    ("What analogy does the textbook use to explain RAG?", {295}),
    ("How does the textbook add randomness to retry waits?", {357}),
    ("What does FIFO mean for conversation history?", {345}),
    ("Where should the unchanging part of a prompt go so caching works?", {358}),
    ("When should a person approve an agent's action?", {355}),
    ("How does Graph RAG find groups of related entities?", {305}),
    ("Which chunking method splits where the topic changes?", {302}),
    ("Which frameworks are compared in Table 18.1?", {359}),
    ("What is the difference between episodic and semantic memory?", {321, 322}),
    ("What does an evaluator-optimizer workflow do?", {370, 371}),
    ("How does the harness notice an agent repeating the same tool call?", {357, 363}),
]

## 1. Documents and chunking

| Ours | LangChain |
|---|---|
| `{"text": ..., "source": ..., "page": ...}` | `Document(page_content=..., metadata={...})` |
| `chunk_page()` (Lab 1) | `RecursiveCharacterTextSplitter` |

In [ ]:
from langchain_core.documents import Document

reader = PdfReader("hitchhikers-guide-v1.pdf")
HEADER = "H. Roitman — The Hitchhiker’s Guide to Agentic AI: From Foundations to Systems\n"
book = [Document(page_content=reader.pages[n - 1].extract_text().replace(HEADER, ""),
                 metadata={"source": "hitchhikers-guide-v1.pdf", "page": n})
        for n in range(292, 375)]
print(len(book), "pages ·", book[3].metadata)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def mean_tokens(docs):
    return round(np.mean([n_tokens(d.page_content) for d in docs]))

as_written = RecursiveCharacterTextSplitter(
    chunk_size=512,          # the book: "tokens per chunk"
    chunk_overlap=64,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
).split_documents(book)
print("as written:", len(as_written), "chunks · mean", mean_tokens(as_written), "tokens")

In [ ]:
# ✏️ TODO 1 (▶ video 2:15): make the splitter measure length in Gemma tokens (a function we already have)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    length_function=___,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(book)
print("Gemma tokens:", len(chunks), "chunks · mean", mean_tokens(chunks), "tokens ·", chunks[5].metadata)

## 2. Embeddings and the vector store

| Ours (Lab 1) | LangChain |
|---|---|
| `embed()` | `OpenAIEmbeddings` (works with any OpenAI-style API, OpenRouter included) |
| the `E` matrix + `search()` | `InMemoryVectorStore` + `.as_retriever()` |

In [ ]:
from langchain_openai import OpenAIEmbeddings

emb = OpenAIEmbeddings(
    model=EMBED_MODEL, base_url=OPENROUTER, api_key=load_key(),
    check_embedding_ctx_length=False,
    model_kwargs={"extra_body": EMBED_EXTRA},
    chunk_size=64, max_retries=5,
)

car, auto, banana = np.array(emb.embed_documents(["car", "automobile", "banana"]))
print("car·automobile", round(car @ auto, 2), "· car·banana", round(car @ banana, 2))

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

store = InMemoryVectorStore.from_documents(chunks, emb)
dense = store.as_retriever(search_kwargs={"k": 3})
for d in dense.invoke("What analogy does the textbook use to explain RAG?"):
    print(d.metadata["page"], d.page_content[:70].replace("\n", " "))

In [ ]:
def hit3(name, retriever):
    hits, context = 0, []
    for q, gold in TESTS:
        top3 = retriever.invoke(q)[:3]
        # ✏️ TODO 2 (▶ video 4:20): it's a hit if the metadata page of any Document in top3 is in gold
        hits += ___
        context.append(sum(n_tokens(d.page_content) for d in top3))
    print(f"{name:26s} hit@3 {hits:2d}/{len(TESTS)} · context ≈ {np.mean(context):,.0f} tokens")

hit3("dense", dense)

## 3. Our BM25 in LangChain + EnsembleRetriever

LangChain has a `BM25Retriever` in `langchain-community`, which is being sunset, and by default it splits words on spaces only
(`"PLAID?"` keeps its question mark, nothing is lowercased) — we tried it: only 4 out of 20.
So we plug in Lab 2's BM25 ourselves: a LangChain retriever is a class with **one method**

| Ours (Lab 2) | LangChain |
|---|---|
| `bm25_rank()` | `class OurBM25(BaseRetriever)` |
| `rrf()` (Listing 16.8) | `EnsembleRetriever` |

In [ ]:
from langchain_core.retrievers import BaseRetriever

def words(text):
    return re.findall(r"\w+", text.lower())

class OurBM25(BaseRetriever):
    docs: list
    k: int = 3

    def _get_relevant_documents(self, query, *, run_manager=None):
        bags = [Counter(words(d.page_content)) for d in self.docs]
        lens = [sum(bag.values()) for bag in bags]
        N, avgdl = len(bags), sum(lens) / len(bags)
        df = Counter(w for bag in bags for w in bag)
        k1, b = 1.5, 0.75
        scores = []
        for bag, dl in zip(bags, lens):
            s = 0.0
            for t in words(query):
                idf = math.log((N - df[t] + 0.5) / (df[t] + 0.5) + 1)
                s += idf * bag[t] * (k1 + 1) / (bag[t] + k1 * (1 - b + b * dl / avgdl))
            scores.append(s)
        # ✏️ TODO 3 (▶ video 5:21): sort the indices by score, high to low, keep the top k, and return their Documents
        best = ___
        return ___

bm25 = OurBM25(docs=chunks, k=3)
hit3("BM25 (ours, in LangChain)", bm25)

In [ ]:
# ✏️ TODO 4 (▶ video 5:38): merge bm25 and dense with EnsembleRetriever, equal weights
from langchain_classic.retrievers import EnsembleRetriever

hybrid = ___
hit3("hybrid, 3 per list", hybrid)

In [ ]:
bm25.k = 60
dense.search_kwargs["k"] = 60
hit3("hybrid, 60 per list", hybrid)

## 4. ParentDocumentRetriever

| Ours (Lab 3) | LangChain |
|---|---|
| cut children from parents, `parent_of` | `child_splitter` + `docstore` |
| `dict.fromkeys(...)` maps children to parents | done for you inside |
| parents = the book's sections | `parent_splitter` (none = the whole Document, one page) |

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore

parent_child = ParentDocumentRetriever(
    vectorstore=InMemoryVectorStore(emb),
    docstore=InMemoryStore(),
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=128, chunk_overlap=16, length_function=n_tokens),
    search_kwargs={"k": 10},
)
parent_child.add_documents(book)
hit3("parent-child (page parents)", parent_child)

## 5. Join it into a chain and ask Gemma

| Ours (Lab 1) | LangChain |
|---|---|
| `SYSTEM_PROMPT` + `build_rag_prompt()` | `ChatPromptTemplate` |
| `ask()` | `ChatOpenAI` |
| `rag()` | everything joined with `\|` |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

SYSTEM_PROMPT = """You are a helpful assistant. Answer the question using ONLY
the provided context. If the context does not contain enough information,
say so explicitly. Cite your sources using [Doc N] notation."""
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"),
])

llm = ChatOpenAI(model=MODEL, base_url=OPENROUTER, api_key=load_key(),
                 temperature=0, extra_body=EXTRA, max_retries=5)

def format_docs(docs):
    return "\n\n".join(
        f"[Doc {i+1}] (Source: {d.metadata['source']}, Page: {d.metadata['page']})\n{d.page_content}"
        for i, d in enumerate(docs[:3]))

In [ ]:
# ✏️ TODO 5 (▶ video 8:00): join prompt → llm → StrOutputParser with the | operator
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def make_rag(retriever):
    return ({"context": retriever | format_docs, "question": RunnablePassthrough()}
            | ___)

In [ ]:
dense.search_kwargs["k"] = 3
rag_dense = make_rag(dense)
for q in ["What analogy does the textbook use to explain RAG?",
          "How do I combine keyword search and embedding search?"]:
    print("Q:", q, "· pages", [d.metadata["page"] for d in dense.invoke(q)])
    print("A:", rag_dense.invoke(q), "\n")

In [ ]:
dense.search_kwargs["k"] = 60
rag_hybrid = make_rag(hybrid)
for q in ["What analogy does the textbook use to explain RAG?",
          "How do I combine keyword search and embedding search?"]:
    print("Q:", q, "· pages", [d.metadata["page"] for d in hybrid.invoke(q)[:3]])
    print("A:", rag_hybrid.invoke(q), "\n")

## Summary

| Ours (Labs 1–3) | LangChain 1.x | Watch out |
|---|---|---|
| a page dict | `Document` | — |
| `chunk_page()` | `RecursiveCharacterTextSplitter` | `length_function=len` counts **characters** (Listing 16.2) |
| `embed()` | `OpenAIEmbeddings` | needs `check_embedding_ctx_length=False` for non-OpenAI models |
| `E` + `search()` | `InMemoryVectorStore.as_retriever()` | — |
| `bm25_rank()` | `class OurBM25(BaseRetriever)` | community's `BM25Retriever` splits on spaces |
| `rrf()` | `EnsembleRetriever` | merges only what each side returns (3 → 13/20, 60 → 17/20) |
| parent-child | `ParentDocumentRetriever` | what a parent is, is still our decision |
| `rag()` | `{...} \| prompt \| llm \| StrOutputParser()` | — |

**Exercises**
1. Give `ParentDocumentRetriever` a `parent_splitter=splitter` (512 tokens) — how do hit@3 and the context change?
2. Try `weights=[0.3, 0.7]` in `EnsembleRetriever` — which questions change?
3. Turn Lab 3's section chunks (`structure_chunks`) into `Document`s and use them with `InMemoryVectorStore`

**Next:** the EP10 recap